# P06 · Перенос обучения и ML-системы / Transfer learning and ML systems

Практика в группе: выполните пример и задания ниже. Отдельной оценки за практику нет. / Group practice: run the example and complete the exercises below. No separate practice grade.

## Данные и протокол / Data and protocol
**CIFAR-10**: реальные цветные изображения 32×32, 10 классов. Объект — одно изображение; признаки — RGB-пиксели, доступные при запросе; цель — категория объекта. / Real 32×32 colour images in 10 classes. One observation is one image; RGB pixels are available at prediction time; the target is the object category.
Источник / Source: https://www.cs.toronto.edu/~kriz/cifar.html (Krizhevsky, 2009). Официальные части: 50 000 train, 10 000 test. / Official partitions: 50,000 train and 10,000 test.
Train и Validation берём без пересечений из официального train; Test — отдельная фиксированная подвыборка official test. Метки Test используются только для фиксированного стратифицированного отбора и финальной оценки. / Train and Validation are disjoint subsets of official train; Test is a fixed stratified subset of official test. Test labels are used only for predefined stratified sampling and final scoring.
Нет ID источника/автора или времени: нельзя утверждать перенос на новых авторов либо будущие данные. Маленькая учебная подвыборка даёт шумную оценку, не результат на полном benchmark. Возможное пересечение источников ImageNet и CIFAR здесь не проверено. / No source/writer/time IDs: this does not establish generalization to unseen authors or future data. Small-subset scores are noisy, not full-benchmark results. Source overlap between ImageNet and CIFAR has not been audited here.
Первый запуск скачивает CIFAR-10 (~163 MiB) и веса (~10 MiB). Интернет нужен для скачивания; платного API нет. CPU достаточно; извлечение признаков может занять несколько минут. / First run downloads data (~163 MiB) and weights (~10 MiB). Internet is needed for downloads; no paid API. CPU is sufficient; feature extraction may take several minutes.


In [ ]:
# В Colab библиотеки обычно установлены. Если import не проходит, выполните эту ячейку и перезапустите среду.
# Colab normally includes these packages. If imports fail, run this cell and restart the runtime.
# %pip install torch torchvision scikit-learn pandas matplotlib pillow


In [ ]:
import copy, json, time, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torchvision, sklearn
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
torch.set_num_threads(2)
print({'python':platform.python_version(), 'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':np.__version__, 'sklearn':sklearn.__version__})


In [ ]:
official_train = CIFAR10('data', train=True, download=True)
official_test = CIFAR10('data', train=False, download=True)
classes = list(official_train.classes)
y_source = np.asarray(official_train.targets)
y_test_source = np.asarray(official_test.targets)
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
backbone = mobilenet_v3_small(weights=weights)
backbone.classifier = nn.Identity()  # 576 признаков вместо 1000 ImageNet-классов / 576 features, not ImageNet labels
backbone.requires_grad_(False)
backbone.eval()
assert all(not p.requires_grad for p in backbone.parameters())
print(preprocess)


## Пример: замороженный MobileNetV3 / Worked example: frozen MobileNetV3
Обучаем только sklearn-классификатор. Backbone остаётся неизменным. / Train only the sklearn head; the backbone stays unchanged.

In [ ]:
def split_indices(train_labels, test_labels, n_train, n_val, n_test, seed):
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for label in range(10):
        ids = rng.permutation(np.flatnonzero(np.asarray(train_labels) == label))
        tids = rng.permutation(np.flatnonzero(np.asarray(test_labels) == label))
        if len(ids) < n_train+n_val or len(tids) < n_test:
            raise ValueError('Not enough observations')
        tr.extend(ids[:n_train]); va.extend(ids[n_train:n_train+n_val]); te.extend(tids[:n_test])
    return tuple(np.asarray(x,dtype=np.int64) for x in (tr,va,te))
def extract_features(dataset, ids, model, transform, batch_size=32):
    class Selected(Dataset):
        def __len__(self): return len(ids)
        def __getitem__(self, i): return transform(dataset[int(ids[i])][0])
    model.eval()
    chunks = []
    with torch.inference_mode():
        for batch in DataLoader(Selected(), batch_size=batch_size, shuffle=False, num_workers=0):
            chunks.append(model(batch).cpu().numpy())
    return np.concatenate(chunks, axis=0)
def fit_head(features, labels, C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=2026)).fit(features, labels)
def choose_run(records):
    # Максимум macro-F1; точное равенство -> меньше C / Max macro-F1, exact ties -> smaller C
    return min(records, key=lambda r: (-r['val_macro_f1'], r['C']))['C']
def predict_image(image, model, transform, head, names):
    image = np.asarray(image)
    if image.shape != (32,32,3) or image.dtype != np.uint8:
        raise ValueError('Expected uint8 RGB image of shape (32,32,3)')
    model.eval()
    with torch.inference_mode():
        features = model(transform(Image.fromarray(image)).unsqueeze(0)).cpu().numpy()
    probabilities = head.predict_proba(features)[0]
    ids = head.classes_.astype(int)
    winner = int(np.argmax(probabilities))
    return {'class_id':int(ids[winner]), 'label':names[int(ids[winner])],
            'probabilities':{names[int(k)]:float(v) for k,v in zip(ids,probabilities)}}
def pixels(dataset, ids):
    return dataset.data[ids].reshape(len(ids),-1).astype(np.float32)/255.0

def report(y, pred):
    return {'accuracy':float(accuracy_score(y,pred)), 'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0))}


In [ ]:
SEED = 2026
train_ids, val_ids, test_ids = split_indices(y_source, y_test_source, 60, 20, 20, SEED)
assert len(set(train_ids)&set(val_ids)) == 0
assert [len(x) for x in (train_ids,val_ids,test_ids)] == [600,200,200]
# Test IDs относятся к другому массиву, числовое совпадение индексов не означает пересечения.
# Test IDs index a separate array: equal integer IDs do not imply overlapping observations.
y_train, y_val, y_test = y_source[train_ids], y_source[val_ids], y_test_source[test_ids]
assert all(set(v)==set(range(10)) for v in (y_train,y_val,y_test))
fig, axes = plt.subplots(2,5,figsize=(10,4))
for k, ax in enumerate(axes.flat):
    ax.imshow(official_train.data[train_ids[np.flatnonzero(y_train==k)[0]]]); ax.set_title(classes[k]); ax.axis('off')
plt.tight_layout(); plt.show()

baseline = fit_head(pixels(official_train,train_ids),y_train,0.1)
print('Pixel baseline validation:', report(y_val,baseline.predict(pixels(official_train,val_ids))))
before = {k:v.clone() for k,v in backbone.state_dict().items()}
start = time.perf_counter()
F_train = extract_features(official_train,train_ids,backbone,preprocess)
F_val = extract_features(official_train,val_ids,backbone,preprocess)
feature_seconds = time.perf_counter()-start
assert F_train.shape == (600,576) and F_val.shape == (200,576)
assert np.isfinite(F_train).all() and np.isfinite(F_val).all()
assert all(torch.equal(before[k],v) for k,v in backbone.state_dict().items())
print('Feature extraction seconds:',feature_seconds)
records, heads = [], {}
for C in [0.01,0.1,1.0]:
    heads[C] = fit_head(F_train,y_train,C)
    records.append({'C':C, 'val_macro_f1':report(y_val,heads[C].predict(F_val))['macro_f1']})
display(pd.DataFrame(records))
selected_C = choose_run(records)
head = heads[selected_C]
selection = {'C':selected_C,'criterion':'validation macro-F1; exact tie -> smaller C',
             'weights':'MobileNet_V3_Small_Weights.IMAGENET1K_V1','seed':SEED}
print('FROZEN BEFORE TEST:', selection)

# Test доступен только после фиксации выбора / Access Test only after freezing selection
F_test = extract_features(official_test,test_ids,backbone,preprocess)
test_results = {'pixels':report(y_test,baseline.predict(pixels(official_test,test_ids))),
                'transfer':report(y_test,head.predict(F_test))}
print('FINAL TEST:',test_results)
ConfusionMatrixDisplay(confusion_matrix(y_test,head.predict(F_test)),display_labels=classes).plot(xticks_rotation=60)
plt.show()

image = official_train.data[val_ids[0]]
response = predict_image(image,backbone,preprocess,head,classes)
assert response['class_id'] == int(head.predict(F_val[:1])[0])
assert np.allclose(list(response['probabilities'].values()),head.predict_proba(F_val[:1])[0],atol=1e-5)
assert np.isclose(sum(response['probabilities'].values()),1)
print('Service response:',response)
# Прогрев, затем 10 полных вызовов на Validation / Warm-up, then 10 end-to-end validation calls
predict_image(image,backbone,preprocess,head,classes)
timings=[]
for i in range(10):
    t=time.perf_counter(); predict_image(official_train.data[val_ids[i]],backbone,preprocess,head,classes)
    timings.append((time.perf_counter()-t)*1000)
print('CPU end-to-end median ms:',float(np.median(timings)))
experiment_log = {**selection,'versions':{'torch':torch.__version__,'torchvision':torchvision.__version__,'sklearn':sklearn.__version__},
                  'train_ids':train_ids.tolist(),'val_ids':val_ids.tolist(),'test_ids':test_ids.tolist(),
                  'class_names':classes,'preprocess':str(preprocess),'test_results':test_results,
                  'median_latency_ms':float(np.median(timings)), 'latency_scope':'warm CPU preprocess+backbone+head, batch=1'}
Path('experiment_hw06.json').write_text(json.dumps(experiment_log,ensure_ascii=False,indent=2))


## Задания студентам / Student exercises
1. Покажите counts по классам для всех частей. Объясните, почему индексы Train и Test могут совпадать численно. / Show class counts for all splits. Explain why Train and Test IDs may share integer values.
2. Проверьте одну строку F_val независимым одиночным проходом. Почему shuffle=False важен при сопоставлении признаков и меток? / Verify one F_val row with a separate forward pass. Why does shuffle=False matter for feature-label alignment?
3. Используя только Validation, найдите ошибку transfer-модели и сравните с baseline на том же изображении. Если ошибок нет — сообщите это. / On Validation only, inspect one transfer-model error and compare the baseline on that image. Report explicitly if none exists.
4. Передайте сервису grayscale, float RGB и неправильный размер. Для каждого ожидается ValueError. Почему уверенный ответ здесь хуже явного отказа? / Pass grayscale, float RGB and wrong-size inputs; each should raise ValueError. Why is a confident response worse than explicit rejection here?
5. Измерьте отдельно feature extraction, head.predict_proba и полный запрос. Почему время head не характеризует задержку всей системы? / Measure feature extraction, head.predict_proba and a full request separately. Why is head-only time insufficient?
**Не выбирайте новые C по уже увиденному Test.** Новый эксперимент требует заранее заданного протокола и нового независимого финального набора. / Do not choose new C values using observed Test results. A new experiment needs a prespecified protocol and a fresh independent final evaluation set.


In [ ]:
# Ваш код и ответы / Your code and answers
